In [3]:
import duckdb
from pathlib import Path

def logical_compare_parquet(dir1, dir2):
    dir1 = str(Path(dir1))
    dir2 = str(Path(dir2))
    con = duckdb.connect()

    # List .parquet files in both dirs
    names1 = {Path(p).name for (p,) in con.execute(f"SELECT * FROM glob('{dir1}/*.parquet')").fetchall()}
    names2 = {Path(p).name for (p,) in con.execute(f"SELECT * FROM glob('{dir2}/*.parquet')").fetchall()}
    common = sorted(names1 & names2)

    only1 = sorted(names1 - names2)
    only2 = sorted(names2 - names1)
    print(f"Only in {dir1}: {only1}")
    print(f"Only in {dir2}: {only2}")

    all_ok = True
    for name in common:
        path1 = f"{dir1}/{name}"
        path2 = f"{dir2}/{name}"

        # Read schemas
        s1 = con.execute(f"DESCRIBE SELECT * FROM parquet_scan('{path1}')").fetchdf()
        s2 = con.execute(f"DESCRIBE SELECT * FROM parquet_scan('{path2}')").fetchdf()
        schema_equal = s1[['column_name','column_type']].equals(s2[['column_name','column_type']])

        # Row counts
        n1 = con.execute(f"SELECT COUNT(*) FROM parquet_scan('{path1}')").fetchone()[0]
        n2 = con.execute(f"SELECT COUNT(*) FROM parquet_scan('{path2}')").fetchone()[0]

        # Set difference both ways (assumes rows are comparable; if there’s no stable key,
        # this treats the entire row as a tuple)
        diff12 = con.execute(
            f"SELECT * FROM parquet_scan('{path1}') EXCEPT ALL SELECT * FROM parquet_scan('{path2}')"
        ).fetchone()
        diff21 = con.execute(
            f"SELECT * FROM parquet_scan('{path2}') EXCEPT ALL SELECT * FROM parquet_scan('{path1}')"
        ).fetchone()

        # If EXCEPT returns at least one row, DuckDB returns a row; we only need to know emptiness.
        has_diff12 = diff12 is not None
        has_diff21 = diff21 is not None

        if schema_equal and n1 == n2 and not has_diff12 and not has_diff21:
            print(f"[OK]   {name}: same schema, same rows ({n1} rows).")
        else:
            all_ok = False
            print(f"[DIFF] {name}: schema_equal={schema_equal}, rowsA={n1}, rowsB={n2}, "
                  f"A\\B_diff={has_diff12}, B\\A_diff={has_diff21}")

    if all_ok and not only1 and not only2:
        print("✅ All matching files have identical table contents.")
    else:
        print("❗Content differences found (see above).")

# Example:
path_a = "/data2/processed_datasets/ehr_foundation_data/ohdsi_cumc_deid/ohdsi_cumc_deid_2023q4r3_v3_mapped/task_labels/phenotype_tte/first_event/Ischemic_Stroke"
path_b = "/data2/processed_datasets/ehr_foundation_data/ohdsi_cumc_deid/ohdsi_cumc_deid_2023q4r3_v3_mapped/task_labels/phenotype_tte_first/Ischemic_Stroke"

logical_compare_parquet(path_a,path_b)


Only in /data2/processed_datasets/ehr_foundation_data/ohdsi_cumc_deid/ohdsi_cumc_deid_2023q4r3_v3_mapped/task_labels/phenotype_tte/first_event/Ischemic_Stroke: []
Only in /data2/processed_datasets/ehr_foundation_data/ohdsi_cumc_deid/ohdsi_cumc_deid_2023q4r3_v3_mapped/task_labels/phenotype_tte_first/Ischemic_Stroke: []
[OK]   held_out.parquet: same schema, same rows (49438 rows).
[OK]   train.parquet: same schema, same rows (78888 rows).
[OK]   tuning.parquet: same schema, same rows (19809 rows).
✅ All matching files have identical table contents.


In [15]:
import pandas as pd
from pathlib import Path
label_path = Path("/data/results/zj2398/femr/cumc/phenotype/transformer_mtpp/labels")
for subpath in label_path.glob("*.parquet"):
    label = pd.read_parquet(subpath)
    print(len(label),subpath.name)
    task = subpath.name.split(".")[0]
    feature_path = Path(f"/data/results/zj2398/femr/cumc/phenotype/results/{task}/transformer_mtpp/features_with_label")
    train_file = pd.read_parquet(feature_path/"train.parquet")
    test_file = pd.read_parquet(feature_path/"test.parquet")
    print(len(train_file),train_file)
    print(len(test_file),test_file)
    break

148804 MASLD.parquet
1729       subject_id prediction_time  tte_label  time_to_event  boolean_value  \
0       11967802      2005-10-14          0         4848.0              0   
1       13962855      2014-01-27          0         3355.0              0   
2       12068223      2023-02-27          0          178.0              0   
3       15530829      2018-05-20          0         1886.0              0   
4       14191822      2007-09-29          0         5937.0              0   
...          ...             ...        ...            ...            ...   
1724    13660182      2022-04-28          1          502.0              0   
1725    14191822      2009-03-24          0         5395.0              0   
1726    12891683      2017-04-07          0         2056.0              0   
1727    14740152      2011-01-09          1         2499.0              0   
1728    13637136      2019-08-07          0         1251.0              0   

                                               fe

In [9]:
import pandas as pd
import pickle as pl
from pathlib import Path
label_path = Path("/data/results/zj2398/femr/cumc/phenotype/transformer_mtpp/features")
for subpath in label_path.glob("*.pkl"):
    with open(subpath, 'rb') as file:
        # Load the data from the file
        feature = pl.load(file)
    print(len(feature['subject_ids']))

443145
327483
280916
291523


In [3]:
import pandas as pd
from pathlib import Path
task = "Ischemic_Stroke"
dir_path = Path("/data/results/zj2398/femr/cumc_0.1/phenotype/mtpp_shared/labels")
sinlge_parquet = pd.read_parquet(dir_path/f"{task}.parquet")
multiple_parquet = pd.read_parquet(dir_path/f"{task}_multiple.parquet")
print(sinlge_parquet)
print(multiple_parquet)


# cols = sorted(ami_parquet.columns)
only_in_single = sinlge_parquet.merge(multiple_parquet.drop_duplicates(), how="left", indicator=True).query("_merge == 'left_only'").drop(columns="_merge")
# Rows only in b
only_in_multiple = multiple_parquet.merge(sinlge_parquet.drop_duplicates(), how="left", indicator=True).query("_merge == 'left_only'").drop(columns="_merge")
print(len(only_in_single ), len(only_in_multiple ))

        subject_id prediction_time  boolean_value     split  tte_label  \
0         15181295      2012-11-29              0  held_out          0   
1          5681528      2023-04-06              0  held_out          0   
2          7834614      2009-12-08              0  held_out          0   
3          9116249      1997-09-13              0  held_out          0   
4          4675822      2011-05-01              0  held_out          0   
...            ...             ...            ...       ...        ...   
148130     9124349      2017-02-09              0     train          0   
148131     7742238      2022-11-05              0     train          0   
148132    14412776      2022-01-03              0     train          0   
148133     6262016      2013-08-08              0     train          1   
148134     7181737      2006-03-12              0     train          0   

        time_to_event  
0              4020.0  
1               194.0  
2              5122.0  
3              

In [8]:
cumc_split_path = "/data2/processed_datasets/zj2398/femr/cumc/mtpp_vocab_all/cumc_split.csv"
# cumc_split = pd.read_csv(cumc_split_path)
# print(cumc_split["split"].unique())
# train_ids = cumc_split

# for disease in ["AMI"]:
#     train_path =  f"/data2/processed_datasets/ehr_foundation_data/ohdsi_cumc_deid/ohdsi_cumc_deid_2023q4r3_v3_mapped/task_labels/phenotype_tte/first_event/{disease}/train.parquet"
#     tuning_path =  f"/data2/processed_datasets/ehr_foundation_data/ohdsi_cumc_deid/ohdsi_cumc_deid_2023q4r3_v3_mapped/task_labels/phenotype_tte/first_event/{disease}/tuning.parquet"
#     held_path =  f"/data2/processed_datasets/ehr_foundation_data/ohdsi_cumc_deid/ohdsi_cumc_deid_2023q4r3_v3_mapped/task_labels/phenotype_tte/first_event/{disease}/held_out.parquet"

In [11]:
import pandas as pd
import pyarrow.parquet as pq

def build_split_indexes(cumc_split_path: str):
    # Read only needed cols, and use compact dtypes
    split = pd.read_csv(
        cumc_split_path,
        usecols=["subject_id", "split_name"],
        dtype={"subject_id": "int64", "split_name": "category"},
    )
    train_idx = pd.Index(split.loc[split["split_name"].eq("train"), "subject_id"].values)
    test_idx  = pd.Index(split.loc[split["split_name"].eq("test"),  "subject_id"].values)
    return train_idx, test_idx

def find_not_belong_subject_ids(
    parquet_path: str,
    allowed_idx: pd.Index,
    batch_size: int = 2_000_000,
):
    """
    Streams subject_id from parquet and returns a set of subject_ids not in allowed_idx.
    If the mismatch set could be huge, you can modify this to write to disk instead.
    """
    pf = pq.ParquetFile(parquet_path)
    not_belong = set()

    for batch in pf.iter_batches(columns=["subject_id"], batch_size=batch_size):
        arr = batch.column(0).to_numpy(zero_copy_only=False)

        # Fast membership: get_indexer returns -1 when not found
        missing = arr[allowed_idx.get_indexer(arr) == -1]

        # Dedup within-batch before adding to python set (faster & smaller)
        if missing.size:
            not_belong.update(pd.unique(missing))

    return not_belong

def check_cohort_splits(train_path: str, tuning_path: str, held_path: str, cumc_split_path: str):
    train_idx, test_idx = build_split_indexes(cumc_split_path)

    bad_train  = find_not_belong_subject_ids(train_path,  train_idx)
    bad_tuning = find_not_belong_subject_ids(tuning_path, train_idx)
    bad_held   = find_not_belong_subject_ids(held_path,   test_idx)

    def report(name, bad_set, show=50):
        if not bad_set:
            print(f"[OK] {name}: all subject_ids belong to expected standard split.")
        else:
            bad_list = sorted(bad_set)
            print(f"[BAD] {name}: {len(bad_list)} subject_ids do NOT belong.")
            print(f"  First {min(show, len(bad_list))}: {bad_list[:show]}")

    report("train.parquet (expected standard TRAIN)", bad_train)
    report("tuning.parquet (expected standard TRAIN)", bad_tuning)
    report("held_out.parquet (expected standard TEST)", bad_held)

    # If you truly need *all* IDs printed (can be enormous), uncomment:
    # print("All bad train IDs:", sorted(bad_train))
    # print("All bad tuning IDs:", sorted(bad_tuning))
    # print("All bad held IDs:", sorted(bad_held))

    return bad_train, bad_tuning, bad_held

# Example usage:
cumc_split_path = "/data2/processed_datasets/zj2398/femr/cumc/mtpp_vocab_all/cumc_split.csv"
for disease in ["Ischemic_Stroke"]:
    train_path =  f"/data2/processed_datasets/ehr_foundation_data/ohdsi_cumc_deid/ohdsi_cumc_deid_2023q4r3_v3_mapped/task_labels/phenotype_tte/first_event/{disease}/train.parquet"
    tuning_path =  f"/data2/processed_datasets/ehr_foundation_data/ohdsi_cumc_deid/ohdsi_cumc_deid_2023q4r3_v3_mapped/task_labels/phenotype_tte/first_event/{disease}/tuning.parquet"
    held_path =  f"/data2/processed_datasets/ehr_foundation_data/ohdsi_cumc_deid/ohdsi_cumc_deid_2023q4r3_v3_mapped/task_labels/phenotype_tte/first_event/{disease}/held_out.parquet"
    bad_train, bad_tuning, bad_held = check_cohort_splits(train_path, tuning_path, held_path, cumc_split_path)


[OK] train.parquet (expected standard TRAIN): all subject_ids belong to expected standard split.
[OK] tuning.parquet (expected standard TRAIN): all subject_ids belong to expected standard split.
[OK] held_out.parquet (expected standard TEST): all subject_ids belong to expected standard split.


In [7]:
print(cumc_split["split_name"].unique())

['train' 'test']


In [8]:
import pandas as pd
from pathlib import Path
base_dir = Path("/data2/processed_datasets/ehr_foundation_data/ohdsi_cumc_deid/ohdsi_cumc_deid_2023q4r3_v3_mapped/post_transform_0.1")
# train_path = Path("/data2/processed_datasets/ehr_foundation_data/ohdsi_cumc_deid/ohdsi_cumc_deid_2023q4r3_v3_mapped/post_transform/data/train")
# for i in range(0,10):
#     train_cohort = pd.read_parquet(train_path/f"{i}.parquet")
#     print(train_cohort["subject_id"])
# meta_path = base_dir/"metadata"/"subject_splits.parquet"
# meta_path = base_dir/"metadata"/"vocabulary_cache"/"mapping"/"data.parquet"
id_list = []
data_dir = base_dir/"data"
for subdir in data_dir.iterdir():
    if not subdir.is_dir():
        continue
    for f in subdir.glob("*.parquet"):
        raw_file = pd.read_parquet(f)
        id_list.extend(raw_file["subject_id"].unique())
        # break
        # id_list.extend(raw_file)
    # break
print(len(id_list))
# meta_data = pd.read_parquet(meta_path)
# print(meta_data)

690298


In [3]:
import pandas as pd
from pathlib import Path
held_out_path = Path("/data2/processed_datasets/ehr_foundation_data/ohdsi_cumc_deid/ohdsi_cumc_deid_2023q4r3_v3_mapped/task_labels/phenotype_tte/AMI")
split = "train.parquet"
sample_parquet = pd.read_parquet(held_out_path/split)
print(sample_parquet)

       subject_id prediction_time  boolean_value  split  tte_label  \
0         7173500      2011-07-31              1  train          1   
1         2229279      2001-10-25              1  train          1   
2        15080688      2020-05-30              1  train          1   
3        13847456      2015-12-25              1  train          1   
4         1746373      2004-04-15              1  train          1   
...           ...             ...            ...    ...        ...   
79340    15116103      2018-03-01              0  train          0   
79341     3951087      2011-12-08              0  train          0   
79342      540844      2016-10-07              0  train          0   
79343     5203792      2017-10-29              0  train          0   
79344     1340880      2019-04-02              0  train          0   

       time_to_event  
0               15.0  
1              137.0  
2              170.0  
3              110.0  
4              141.0  
...              ... 

In [1]:
import meds_reader
mimic_data = meds_reader.SubjectDatabase("/shared/share_mala/zj2398/mimic/data/meds_v0.6_reader")
print(len(mimic_data))

364627


In [ ]:
import pickle
tokenizer_path = "/data/models/zj2398/femr/cumc_full/tpp/tokenizer"
ontology_path = "/data/models/zj2398/femr/cumc_full/tpp/ontology.pkl"
with open(ontology_path, 'rb') as f:
    ontology = pickle.load(f)
tokenizer = femr.models.tokenizer.HierarchicalTokenizer.from_pretrained(tokenizer_path, ontology=ontology)

        subject_id prediction_time  boolean_value     split  tte_label  \
0         11459411      2019-08-27              0  held_out          0   
1          8624537      2021-10-03              0  held_out          0   
2          5504403      2020-11-29              0  held_out          0   
3          7135352      2020-12-26              0  held_out          0   
4           812724      2022-11-14              0  held_out          0   
...            ...             ...            ...       ...        ...   
148806    15116103      2018-03-01              0     train          0   
148807     3951087      2011-12-08              0     train          0   
148808      540844      2016-10-07              0     train          0   
148809     5203792      2017-10-29              0     train          0   
148810     1340880      2019-04-02              0     train          0   

        time_to_event  
0              1546.0  
1               819.0  
2              1120.0  
3              